<a href="https://colab.research.google.com/github/alfnihilus/computational-chem/blob/main/creat_molecul.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# First, install python library
!pip install pubchempy rdkit pyopsin ase
!pip install rdkit py3Dmol -q

In [ ]:
# @markdown # Also Run This

!java -version
import requests
import os

def download_opsin_latest():
    if os.path.exists("opsin.jar"):
        print("OPSIN sudah ada, skip download.")
        return

    # Ambil info versi terbaru dari GitHub API
    api_url = "https://api.github.com/repos/dan2097/opsin/releases/latest"
    response = requests.get(api_url)
    data = response.json()

    latest_version = data["tag_name"]
    print(f"Versi OPSIN terbaru: {latest_version}")

    # Cari file JAR di assets
    jar_url = None
    for asset in data["assets"]:
        if asset["name"].endswith("jar-with-dependencies.jar"):
            jar_url = asset["browser_download_url"]
            break

    if not jar_url:
        print("JAR tidak ditemukan di release terbaru!")
        return

    # Download JAR
    print(f"Mendownload dari: {jar_url}")
    jar_data = requests.get(jar_url)
    with open("opsin.jar", "wb") as f:
        f.write(jar_data.content)
    print("Download selesai!")

download_opsin_latest()

In [ ]:
import pubchempy as pcp
import subprocess
import os

# @markdown # Pembuatan SMILES
# @markdown ### Masukkan Nama IUPAC (von Baeyer Style) atau Nama Umum/Trivial (Cth. Etanol, 2,4,6-trinitrotoluene)
input_nama_senyawa = "water"  # @param {type:"string"}

def opsin_name_to_smiles(name, opsin_jar_path="opsin.jar"):
    """Convert IUPAC name to SMILES using OPSIN JAR."""
    if not os.path.exists(opsin_jar_path):
        print(f"OPSIN JAR not found at '{opsin_jar_path}'. Download from: https://github.com/dan2097/opsin/releases")
        return None
    try:
        result = subprocess.run(
            ["java", "-jar", opsin_jar_path],
            input=name,
            capture_output=True,
            text=True,
            timeout=30
        )
        smiles = result.stdout.strip()
        return smiles if smiles else None
    except Exception as e:
        print(f"Error running OPSIN: {e}")
        return None

def cek_smiles_lengkap(nama_atau_rumus):
    # Coba cari di PubChem terlebih dahulu
    results = pcp.get_compounds(nama_atau_rumus, 'name')

    if results:
        print(f"Ditemukan di PubChem untuk: {nama_atau_rumus}")
        for compound in results:
            print(f"Nama         : {compound.iupac_name}")
            print(f"Canonical SMILES : {compound.connectivity_smiles}")
            print(f"Isomeric SMILES  : {compound.smiles}")
            print("-" * 30)
        return

    # Fallback ke OPSIN
    print(f"Tidak ditemukan di PubChem untuk '{nama_atau_rumus}', mencoba membaca via OPSIN...")
    smiles_opsin = opsin_name_to_smiles(nama_atau_rumus, opsin_jar_path="opsin.jar")

    if smiles_opsin:
        print(f"Ditemukan oleh OPSIN untuk: {nama_atau_rumus}")
        print(f"SMILES dari OPSIN: {smiles_opsin}")
    else:
        print(f"Gagal: Nama '{nama_atau_rumus}' tidak dikenali oleh PubChem maupun OPSIN.")

# Memanggil fungsi
cek_smiles_lengkap(input_nama_senyawa)

In [ ]:
import pubchempy as pcp
from rdkit import Chem
from rdkit.Chem import AllChem
from ase import Atoms

def generate_molecule(identifier):
    try:
        results = []
        search_type = ""

        # 1. Coba cari berdasarkan CID jika identifier adalah angka murni
        if identifier.isdigit():
            print(f"Mencari sebagai CID: {identifier}...")
            results = pcp.get_compounds(identifier, 'cid')
            if results:
                search_type = "CID"

        # 2. Jika belum ditemukan, coba cari sebagai SMILES
        #    SMILES seringkali mengandung '=', '#', '(', ')', '[', ']', '.', '@', '/', '\\'
        #    dan tidak boleh hanya angka. Periksa beberapa karakter khas SMILES.
        if not results and (any(c in identifier for c in ['=', '#', '(', ')', '[', ']', '.', '@', '/', '\\']) or (not identifier.isalpha() and not identifier.isdigit())):
            print(f"Tidak ditemukan sebagai CID. Mencoba sebagai SMILES: {identifier}...")
            results = pcp.get_compounds(identifier, 'smiles')
            if results:
                search_type = "SMILES"

        # 3. Jika belum ditemukan, coba cari sebagai Nama
        if not results:
            print(f"Tidak ditemukan sebagai CID/SMILES. Mencoba sebagai Nama: {identifier}...")
            results = pcp.get_compounds(identifier, 'name')
            if results:
                search_type = "Nama"

        # 4. Jika belum ditemukan, coba cari berdasarkan Rumus
        if not results:
            print(f"Tidak ditemukan sebagai CID/SMILES/Nama. Mencoba sebagai Rumus: {identifier}...")
            results = pcp.get_compounds(identifier, 'formula')
            if results:
                search_type = "Rumus"

        if not results:
            print("Molekul tidak ditemukan di database PubChem.")
            return

        # Mengambil hasil pertama yang ditemukan
        comp = results[0]
        smiles = comp.smiles
        print(f"Ditemukan! Tipe: {search_type}, SMILES: {smiles}")

        # 3. Proses RDKit untuk Koordinat 3D
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            print(f"Error: Tidak dapat mengonversi SMILES '{smiles}' menjadi objek molekul RDKit.")
            return

        mol = Chem.AddHs(mol)

        # Memberikan koordinat 3D awal
        # Menggunakan AllChem.ETKDGv2() untuk hasil yang lebih baik
        AllChem.EmbedMolecule(mol, AllChem.ETKDGv2())
        # Optimasi struktur agar stabil (Universal Force Field)
        AllChem.UFFOptimizeMolecule(mol)

        # 4. Cetak koordinat
        print(f"\nKoordinat untuk {identifier} (dari {search_type}):")
        conf = mol.GetConformer()
        for i, atom in enumerate(mol.GetAtoms()):
            pos = conf.GetAtomPosition(i)
            print(f"{atom.GetSymbol()} {pos.x:.4f} {pos.y:.4f} {pos.z:.4f}")

    except Exception as e:
        print(f"Error: {e}")

# Input dari user
# @markdown # Pembuatan Koordinat Secara Otomatis (Sumber PubChem)
# @markdown ### Masukkan Nama atau Rumus Kimia (misal: H2O, Ethanol, C6H6, Flavonoid, 2244, CCCCCCCCCCCC)
user_input ="water" # @param {type:"string"}
generate_molecule(user_input)


In [ ]:
import py3Dmol
from IPython.display import display, HTML
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import AllChem

# @title 🛠️ Multi-Input Molecule Visualizer
# @markdown Masukkan data pada salah satu atau kedua kolom di bawah ini.

# --- INPUT AREA ---
smiles_code = "O" # @param {type:"string"}
xyz_input = """
C 0.0000 0.0000 0.0000
O 1.2000 0.0000 0.0000
"""
# @markdown ---
# @markdown If XYZ, input manually

def add_space(height="20px"):
    """Fungsi untuk memberi jarak vertikal antar elemen."""
    display(HTML(f'<div style="height: {height};"></div>'))

def visualize_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        print(f"❌ Error: Kode SMILES '{smiles}' tidak valid.")
        return

    print(f"✅ HASIL SMILES: {smiles}")

    # 1. Judul & Visualisasi 2D
    print("📍 Struktur 2D (Diagram)")
    img_2d = Draw.MolToImage(mol, size=(300, 300))
    display(img_2d)

    # --- MEMBERI JARAK DI SINI ---
    add_space("5px")

    # 2. Judul & Visualisasi 3D
    print("📍 Struktur 3D Interaktif")
    mol_3d = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol_3d, AllChem.ETKDG())
    AllChem.UFFOptimizeMolecule(mol_3d)
    mb = Chem.MolToMolBlock(mol_3d)

    view = py3Dmol.view(width=500, height=400)
    view.addModel(mb, 'sdf')
    view.setStyle({'stick': {'radius': 0.15}, 'sphere': {'radius': 0.3}})
    view.zoomTo()
    view.show()

def visualize_xyz(xyz_raw):
    lines = [line.strip() for line in xyz_raw.strip().split('\n') if line.strip()]
    if not lines:
        return

    try:
        int(lines[0])
        xyz_final = "\n".join(lines)
    except ValueError:
        num_atoms = len(lines)
        xyz_final = f"{num_atoms}\nGenerated by AI Collaborator\n" + "\n".join(lines)

    print(f"✅ HASIL KOORDINAT XYZ ({len(lines)} atom)")
    add_space("5px")

    view = py3Dmol.view(width=500, height=400)
    view.addModel(xyz_final, 'xyz')
    view.setStyle({'stick': {'radius': 0.15}, 'sphere': {'radius': 0.3}})
    view.zoomTo()
    view.show()

# --- LOGIKA EKSEKUSI ---
has_input = False

if smiles_code.strip():
    visualize_smiles(smiles_code)
    has_input = True
    # Garis pemisah tebal jika ada input berikutnya
    display(HTML('<hr style="border: 2px solid #444; margin: 10px 0;">'))

if xyz_input.strip() and not xyz_input.isspace():
    visualize_xyz(xyz_input)
    has_input = True

if not has_input:
    print("⚠️ Silakan masukkan kode SMILES atau koordinat XYZ terlebih dahulu.")

In [ ]:
# @title SMILES & XYZ from Chemical Formula

import pubchempy as pcp
from rdkit import Chem
from rdkit.Chem import AllChem

def formula_to_structures(formula):
    print(f"--- Memproses Rumus: {formula} ---")

    # 1. Cari SMILES dari PubChem berdasarkan rumus
    # Mengambil hasil pertama yang paling relevan
    results = pcp.get_compounds(formula, 'formula')

    if not results:
        return "Maaf, rumus kimia tidak ditemukan di database."

    # Kita ambil senyawa pertama yang ditemukan
    compound = results[0]
    smiles = compound.connectivity_smiles
    name = compound.iupac_name

    print(f"Nama: {name}")
    print(f"SMILES: {smiles}")

    # 2. Konversi SMILES ke Koordinat XYZ menggunakan RDKit
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)  # Tambahkan atom Hidrogen

    # Generate konformasi 3D
    AllChem.EmbedMolecule(mol, AllChem.ETKDG())
    AllChem.MMFFOptimizeMolecule(mol) # Optimasi energi sederhana

    # 3. Format ke XYZ
    num_atoms = mol.GetNumAtoms()
    conf = mol.GetConformer()

    xyz_output = f"{num_atoms}\nGenerated by AI\n"
    for i in range(num_atoms):
        atom = mol.GetAtomWithIdx(i)
        pos = conf.GetAtomPosition(i)
        symbol = atom.GetSymbol()
        xyz_output += f"{symbol:<2} {pos.x:>10.5f} {pos.y:>10.5f} {pos.z:>10.5f}\n"

    return xyz_output

# Contoh penggunaan
formula_input = "H2O" # @param{type:"string"}
# Untuk kristal anorganik, hasilnya adalah unit molekuler terkecil
# formula_input = "C6H6" # Contoh organik
print(formula_to_structures(formula_input))